In [1]:
# Basic libraries 
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pykalman import KalmanFilter
from itertools import combinations
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.stattools import coint
import statsmodels.api as sm
import pickle
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
sys.path.insert(0, PROJECT_ROOT)

In [2]:
# Scripts 
from scripts.src.cointegration import *
from scripts.src.kalman import *
from scripts.src.pairs import *
from scripts.src.trading_signal import *
from scripts.src.backtest import *
from scripts.src.plots import *
from scripts.src.utils import *

In [3]:
# Load in data from previous notebooks
df1_is = pd.read_csv("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/df1_is")
df1_oos = pd.read_csv("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/df1_oos")

df2_is = pd.read_csv("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/df2_is_updated")
df2_oos = pd.read_csv("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/df2_oos_updated")

static_results_df = pd.read_csv("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/static_hedge_ratio")
dynamic_results_df = pd.read_csv("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/dynamic_hedge_ratio")

In [4]:
# Load dictionaries from previous notebook
with open("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/dictionaries/static_spreads.pkl", "rb") as f:
    static_spreads = pickle.load(f)
with open("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/dictionaries/static_models.pkl", "rb") as f:
    static_models = pickle.load(f)

with open("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/dictionaries/dynamic_spreads.pkl", "rb") as f:
    dyamic_spreads = pickle.load(f)
with open("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/dictionaries/dynamic_details.pkl", "rb") as f:
    dynamic_details = pickle.load(f)


# Implementation - Trading Signal
- For backtest here and compare the Kalman indicator to the baseline and a simple long strategy of only Asset A and Asset B (backtest.py)
- We implement a basic trading signal using standardised values for our dynamic spreads (z-scores)

The general rule is that:
- If the z-score > +2 - we short the spread 
- If the z-score < -2 - we long the spread
- If the z-score = 0 - we exit our position 

Mathematically, we can express it in a mapping:
$$
\text{Position}(z) =
\begin{cases}
-1 & \text{if } z > 2 \\
+1 & \text{if } z < -2 \\
\text{hold previous position} & \text{otherwise}
\end{cases}
$$

*Where you should add complexity (high ROI) - **Continue here!!!**

Instead of inventing a totally new signal, I’d suggest upgrading your current one in controlled, explainable ways:

1. Add realistic trading logic

Your current mapping:

Enter: z>2 or z<−2
Exit: z=0

This is a bit idealised. You could improve it with:

Exit band (e.g. exit when ∣z∣<0.5)
Holding region (don’t flip too often)

This shows you understand market microstructure + execution.

2. Introduce rolling / dynamic elements

For example:

Rolling mean & std (instead of static)
Or compare:
60-day vs 120-day window

This adds methodological depth without being messy.

3. Risk management layer

Very valuable academically:

Position scaling:

Position∝−z

Or cap exposure:

Position=max(−1,min(1,−z/2))

This signals you understand portfolio construction, not just signals.

4. Backtest evaluation (this matters most)

Spend words here, not inventing signals:

Sharpe ratio
Drawdowns
Turnover
Transaction costs impact

This is where top marks usually come from.*

In [5]:
#  Create the signals using the dynamic hedge ratio inputs (y, x, alpha_t, beta_t)
signals = generate_kalman_signals(dynamic_details)

"""
Signal indicators:
-1 = Short spread
1 = Long spread
0 = Exit position
"""

signals

{('700 HK Equity vs 1347 HK Equity',
  0.5):              y         x   alpha_t    beta_t  spread_t  obs_cov    zscore  \
 0     4.597148  2.175547  0.737549  1.604573  0.368775      0.5  4.088959   
 1     4.603339  2.162173  0.772797  1.675708  0.207371      0.5  2.239234   
 2     4.659024  2.188632  0.783063  1.703784  0.147003      0.5  1.547406   
 3     4.695038  2.178495  0.791309  1.722627  0.150994      0.5  1.593144   
 4     4.718079  2.183576  0.796288  1.735401  0.132412      0.5  1.380179   
 ...        ...       ...       ...       ...       ...      ...       ...   
 1473  6.207606  3.808062  4.174089  0.568151 -0.130036      0.5 -1.627530   
 1474  6.138742  3.793667  4.176448  0.564741 -0.180147      0.5 -2.201818   
 1475  6.160659  3.816833  4.178816  0.561604 -0.161706      0.5 -1.990484   
 1476  6.213882  3.820105  4.180271  0.559606 -0.104144      0.5 -1.330803   
 1477  6.221892  3.775653  4.180828  0.558433 -0.067384      0.5 -0.909529   
 
       position  


## In-Sample Backtest

## OOS Backtest